# Part3 Stage1 Difix Dataset Prep

这个 notebook 用于调用 `part3_BRPO/scripts/prepare_stage1_difix_dataset.py`。
默认示例是 **Re10k-1 + RegGS + midpoint**。你只需要先改配置 cell，再按顺序运行 `select -> difix -> pack`。


In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

def run_cmd(cmd, env=None):
    print(' '.join(shlex.quote(str(x)) for x in cmd))
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    subprocess.run([str(x) for x in cmd], check=True, env=full_env)


In [ ]:
# ===== Config =====
PROJECT_ROOT = Path('/home/bzhang512/CV_Project')
SCRIPT_PATH = PROJECT_ROOT / 'part3_BRPO/scripts/prepare_stage1_difix_dataset.py'
CONDA_RUN = Path('/home/bzhang512/miniconda3/bin/conda')
CONDA_ENV = 'reggs'
HF_ENV = {'HF_ENDPOINT': 'https://hf-mirror.com'}

SCENE_NAME = 'Re10k-1'
BACKEND = 'reggs'  # reggs | s3po
RUN_KEY = 're10k1__reggs__midpoint__v1'
PLACEMENT = 'midpoint'  # midpoint | tertile | both | manual
MANUAL_IDS = ''  # e.g. '17,52,88'

TRAIN_DIR = PROJECT_ROOT / 'output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check/train'
RENDER_DIR = PROJECT_ROOT / 'output/part2/re10k_1/reggs_re10k1_re10k-ckpt_sr50_nv9_sm2_comparison_check/test_all_non_train_subset_v1'

DIFIX_MODEL_NAME = 'nvidia/difix_ref'
HEIGHT = 512
WIDTH = 512
TIMESTEP = 199
PROMPT = 'remove degradation'

RUN_ROOT = PROJECT_ROOT / f'dataset/{SCENE_NAME}/part3_stage1/{RUN_KEY}'
print('SCRIPT_PATH =', SCRIPT_PATH)
print('RUN_ROOT =', RUN_ROOT)


In [ ]:
# ===== select =====
cmd = [
    CONDA_RUN, 'run', '-n', CONDA_ENV, 'python', SCRIPT_PATH,
    '--stage', 'select',
    '--backend', BACKEND,
    '--scene-name', SCENE_NAME,
    '--train-dir', TRAIN_DIR,
    '--render-dir', RENDER_DIR,
    '--run-key', RUN_KEY,
    '--placement', PLACEMENT,
]
if MANUAL_IDS.strip():
    cmd.extend(['--manual-ids', MANUAL_IDS])
run_cmd(cmd)


In [ ]:
# ===== difix =====
cmd = [
    CONDA_RUN, 'run', '-n', CONDA_ENV, 'python', SCRIPT_PATH,
    '--stage', 'difix',
    '--backend', BACKEND,
    '--scene-name', SCENE_NAME,
    '--run-key', RUN_KEY,
    '--difix-model-name', DIFIX_MODEL_NAME,
    '--height', str(HEIGHT),
    '--width', str(WIDTH),
    '--timestep', str(TIMESTEP),
    '--prompt', PROMPT,
]
run_cmd(cmd, env=HF_ENV)


In [ ]:
# ===== pack =====
cmd = [
    CONDA_RUN, 'run', '-n', CONDA_ENV, 'python', SCRIPT_PATH,
    '--stage', 'pack',
    '--backend', BACKEND,
    '--scene-name', SCENE_NAME,
    '--run-key', RUN_KEY,
]
run_cmd(cmd)


In [ ]:
# ===== optional quick check =====
import json
manifest = RUN_ROOT / 'manifests/pack_manifest.json'
if manifest.exists():
    print(json.loads(manifest.read_text()))
else:
    print('pack manifest not found yet:', manifest)
